# Maia Corpus v9 — Mathematics Acquisition Engine v1.0
**Maia Lite 0.4.0 | MaiaFormer-355 | Colab CPU**

Meta: 600 milhões de tokens proxy (3,5 caracteres/token). 450 M de textos científicos com licença explícita por registro; até 150 M de problemas e soluções sintéticas (separados). Não usa FineMath nem OpenWebMath. Não altera code_v10, code_v11 ou Gutenberg. As cotas temáticas são heurísticas, não uma classificação científica validada. Execute todas as células; após interrupção execute novamente. Apenas uma sessão por diretório. Não executar conteúdo dos documentos.

Fontes: https://huggingface.co/datasets/PleIAs/common_corpus e https://huggingface.co/datasets/nvidia/OpenMathInstruct-2 (CC BY 4.0 declarada; derivados de GSM8K/MATH e Llama 3.1). Soluções sintéticas não são verificadas matematicamente; manter atribuição e avaliar condições upstream antes da publicação. Candidatos não equivalem a aprovação jurídica.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub','pyarrow','fsspec','aiohttp'])

In [ ]:
from pathlib import Path
import json,gzip,hashlib,os,re,time,random,collections,unicodedata
import pyarrow.parquet as pq
import fsspec,aiohttp
from huggingface_hub import HfApi,hf_hub_url
BASE=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
OUT=BASE/'math_v10'; OUT.mkdir(parents=True,exist_ok=True)
BATCH=OUT/'batches'; BATCH.mkdir(exist_ok=True)
TARGET={'algebra':120_000_000,'analysis':120_000_000,'statistics':90_000_000,'applied':120_000_000,'problems':150_000_000}
RATIO=3.5
ALLOW={'cc-by-4.0','cc-by-3.0','cc0-1.0','public-domain','public domain','cc-by-2.0','cc-by-2.5'}
REPOS={'science':'PleIAs/common_corpus','problems':'nvidia/OpenMathInstruct-2'}
MAX_FILES=10000
SEED=2026
# Enable only if synthetic material is desired; separate from natural text.
ENABLE_SYNTHETIC=True
CONFIG={'target':TARGET,'ratio':RATIO,'allow':sorted(ALLOW),'repos':REPOS,'seed':SEED,'synthetic':ENABLE_SYNTHETIC,'policy':'math-v1'}
def digest(t):return hashlib.sha256(t.encode('utf8')).hexdigest()
def atomic(p,obj):
 tmp=p.with_suffix(p.suffix+'.part'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8'); os.replace(tmp,p)
def normalize(t):return ' '.join(unicodedata.normalize('NFC',t).split())
def normhash(t):return digest(normalize(t))
print(OUT, sum(TARGET.values()))

In [ ]:
# Classificação conservadora por vocabulário matemático EN/PT/ES.
PAT={
'algebra':r'\b(?:algebra|álgebra|matrix|matrices|matriz|eigenvalues?|vector spaces?|linear transformations?|group theory|polynomial|polinômio|polinomio|determinant|determinante|ring theory|field theory|espaço vetorial)\b',
'analysis':r'\b(?:calculus|cálculo|derivative|derivada|integral|integrals|integration|convergence|convergência|convergencia|continuity|continuidade|continuidad|real analysis|complex analysis|fourier|taylor series|measure theory|functional analysis)\b',
'statistics':r'\b(?:probability|probabilidade|probabilidad|random variable|variável aleatória|variable aleatoria|bayesian|bayes|variance|variância|varianza|stochastic|estocástico|statistical inference|confidence interval|hypothesis testing|regression|regressão|regresión)\b',
'applied':r'\b(?:numerical methods?|métodos numéricos|finite elements?|elementos finitos|finite differences?|differential equations?|equações diferenciais|ecuaciones diferenciales|optimization|otimização|optimización|runge.kutta|numerical analysis|boundary value|initial value|mathematical model|modelagem matemática|modelización matemática|finite volume)\b'}
PAT={k:re.compile(v,re.I) for k,v in PAT.items()}
FORM=re.compile(r'\\(?:frac|sum|int|begin|partial|nabla|sqrt|lim|alpha|beta|mathbf)\b|[∫∑∂∇√≤≥]|\$[^\n$]{2,200}\$')
def topic(t,title=''):
 scores={k:len(set(x.lower() for x in p.findall(t)))+2*len(p.findall(title)) for k,p in PAT.items()}
 k=max(scores,key=scores.get)
 if scores[k]<3 or (not FORM.search(t) and scores[k]<5):return None
 return k

def licenses(x):
 a=[x] if isinstance(x,str) else x if isinstance(x,list) else []
 return [v.strip().lower() for v in a] if a and all(isinstance(v,str) and v.strip().lower() in ALLOW for v in a) else []
def science_meta(r):
 if re.sub(r'[ _-]','',str(r.get('open_type','')).lower())!='openscience':return None
 if not isinstance(r.get('identifier'),str) or not r['identifier'].strip():return None
 if str(r.get('language','')).lower() not in {'en','pt','es','eng','por','spa','english','portuguese','spanish'}:return None
 return licenses(r.get('license')) or None

def quality(t):
 return isinstance(t,str) and 300<=len(t)<=1_000_000 and '\x00' not in t and t.count('\ufffd')<=2
assert topic(r'Linear algebra matrix eigenvalues vector spaces $x+y$')=='algebra'
assert licenses('CC-BY-4.0')==['cc-by-4.0'] and not licenses('unknown')
assert normhash('a  b')==normhash('a b')

In [ ]:
# Plano fixado por SHA do Hugging Face e README arquivado para atribuição.
from huggingface_hub import hf_hub_download
planpath=OUT/'plan.json'
if planpath.exists():
 plan=json.loads(planpath.read_text()); assert plan['config']==CONFIG, 'Configuração mudou: use outro diretório.'
else:
 plan={'config':CONFIG,'sources':{}}
 for kind,repo in REPOS.items():
  if kind=='problems' and not ENABLE_SYNTHETIC:continue
  info=HfApi().dataset_info(repo,files_metadata=True,timeout=60)
  files=[x.rfilename for x in info.siblings if x.rfilename.endswith('.parquet') and (kind!='problems' or re.search(r'/train-\d+-of-\d+\.parquet$',x.rfilename))]
  assert files, f'Nenhum parquet compatível: {repo}'
  random.Random(SEED).shuffle(files)
  readme=Path(hf_hub_download(repo,'README.md',repo_type='dataset',revision=info.sha)).read_text(encoding='utf8')
  if kind=='problems':assert 'cc-by-4.0' in readme.lower(), 'Licença do dataset mudou'
  (OUT/f'{kind}_README.md').write_text(readme,encoding='utf8')
  plan['sources'][kind]={'repo':repo,'revision':info.sha,'files':files,'readme_sha256':digest(readme)}
 atomic(planpath,plan)
print({k:(v['revision'],len(v['files'])) for k,v in plan['sources'].items()})

In [ ]:
# Recuperação transacional: lotes são a única fonte de verdade.
seen=set(); questions=set(); totals=collections.Counter(); counts=collections.Counter(); langs=collections.Counter(); rejects=collections.Counter(); done=set()
for p in sorted(BATCH.glob('*.json.gz')):
 with gzip.open(p,'rt',encoding='utf8') as f:o=json.load(f)
 key=(o['key'],o['start']); assert key not in done; done.add(key)
 for r in o['records']:
  assert digest(r['text'])==r['sha256'],p
  h=normhash(r['text']); assert h not in seen,p; seen.add(h)
  if r.get('question_hash'):
   assert r['question_hash'] not in questions; questions.add(r['question_hash'])
  totals[r['category']]+=r['proxy_tokens']; counts[r['category']]+=1; langs[r['language']]+=r['proxy_tokens']
 rejects.update(o['rejects'])
def report():
 data={'status':'TARGET_MET' if all(totals[k]>=v for k,v in TARGET.items()) else 'PARTIAL_TARGET','proxy_tokens':dict(totals),'target':TARGET,'deficits':{k:max(0,v-totals[k]) for k,v in TARGET.items()},'documents':dict(counts),'tokens_by_language':dict(langs),'rejections':dict(rejects),'completed_groups':len(done),'rights_status':'SOURCE_LICENSE_CANDIDATES_ONLY','requires_final_tokenizer':True,'synthetic_solutions_verified':False}
 atomic(OUT/'summary.json',data); return data
print(json.dumps(report(),indent=2))

In [ ]:
# Aquisição em escala real. Metadados primeiro no Common Corpus.
# Cada grupo é subdividido em lotes de 512 linhas para reduzir RAM e perda em interrupções.
start_time=time.monotonic(); initial=sum(totals.values())
def needed(kind):return any(totals[k]<v for k,v in TARGET.items() if (k=='problems')==(kind=='problems'))
def process(kind,src,file):
 key=src['repo']+'@'+src['revision']+'/'+file
 url=hf_hub_url(src['repo'],file,repo_type='dataset',revision=src['revision'])
 with fsspec.open(url,'rb',block_size=4*2**20,client_kwargs={'timeout':aiohttp.ClientTimeout(total=300,sock_read=90)}) as f:
  pf=pq.ParquetFile(f); cols=set(pf.schema_arrow.names)
  req={'text','open_type','license','identifier','language','title'} if kind=='science' else {'problem','generated_solution','expected_answer','problem_source'}
  assert req<=cols, f'Campos ausentes: {req-cols}'
  for g in range(pf.num_row_groups):
   if not needed(kind):break
   if kind=='science':
    meta=sorted(cols & {'open_type','license','identifier','title','creator','collection','language','curator'})
    rows=pf.read_row_group(g,columns=meta).to_pylist()
    eligible=any(science_meta(r) for r in rows)
    texts=pf.read_row_group(g,columns=['text']).column('text') if eligible else None
   else:rows=pf.read_row_group(g,columns=sorted(req)).to_pylist(); texts=None
   for j in range(0,len(rows),512):
    if not needed(kind):break
    pos=f'{g}:{j}'
    if (key,pos) in done:continue
    records=[]; rej=collections.Counter()
    for i in range(j,min(j+512,len(rows))):
     r=rows[i]; qh=None
     if kind=='science':
      lic=science_meta(r)
      if not lic:rej['metadata_or_license']+=1;continue
      t=texts[i].as_py(); lang=str(r['language']).lower()
      if not quality(t):rej['quality']+=1;continue
      cat=topic(t,str(r.get('title') or ''))
      if not cat:rej['not_math']+=1;continue
      provenance=r; synthetic=False
     else:
      q=r['problem']; sol=r['generated_solution']; ans=r['expected_answer']
      if not all(isinstance(x,str) and x.strip() for x in (q,sol,ans)) or len(q)>6000 or len(sol)>30000:rej['problem_quality']+=1;continue
      # Prefer new augmented problems; avoid repeated original benchmark questions.
      if r['problem_source'] not in {'augmented_math','augmented_gsm8k'}:rej['original_benchmark_problem']+=1;continue
      qh=normhash(q)
      if qh in questions:rej['duplicate_problem']+=1;continue
      t='Problem:\n'+q+'\n\nSolution:\n'+sol
      if not quality(t):rej['quality']+=1;continue
      cat='problems'; lang='en'; lic=['cc-by-4.0']; synthetic=True
      provenance={'problem_source':r['problem_source'],'expected_answer':ans,'license_scope':'dataset_card','generator':'Llama-3.1-405B-Instruct'}
     if totals[cat]>=TARGET[cat]:rej['quota']+=1;continue
     h=normhash(t)
     if h in seen:rej['duplicate']+=1;continue
     rec={'text':t,'sha256':digest(t),'language':lang,'category':cat,'license_evidence':lic,'rights_status':'SOURCE_LICENSE_CANDIDATE','source':src['repo'],'revision':src['revision'],'source_file':key,'row_group':g,'row':i,'characters':len(t),'proxy_tokens':round(len(t)/RATIO),'source_metadata':provenance,'synthetic':synthetic,'question_hash':qh}
     records.append(rec);seen.add(h)
     if qh:questions.add(qh)
     totals[cat]+=rec['proxy_tokens'];counts[cat]+=1;langs[lang]+=rec['proxy_tokens']
    p=BATCH/f'{digest(key)[:24]}_{g:08d}_{j:08d}.json.gz'; tmp=p.with_suffix('.part')
    with gzip.open(tmp,'wt',encoding='utf8',compresslevel=3) as out:json.dump({'key':key,'start':pos,'records':records,'rejects':dict(rej)},out,ensure_ascii=False,default=str)
    os.replace(tmp,p);done.add((key,pos));rejects.update(rej)
   print('GROUP',g,'totals',dict(totals),'proxy/s',round((sum(totals.values())-initial)/max(1,time.monotonic()-start_time)),flush=True);report()
finishedpath=OUT/'finished.json'; finished=set(json.loads(finishedpath.read_text())) if finishedpath.exists() else set()
for kind,src in plan['sources'].items():
 attempted=0
 for file in src['files']:
  if not needed(kind) or attempted>=MAX_FILES:break
  tag=kind+'/'+file
  if tag in finished:continue
  attempted+=1; print('READ',kind,file,flush=True)
  try:process(kind,src,file)
  except Exception as e:
   atomic(OUT/'last_error.json',{'kind':kind,'file':file,'error':str(e)[:1000]})
   raise RuntimeError('Interrupção: execute todas as células para recuperar os lotes persistidos.') from e
  finished.add(tag);atomic(finishedpath,sorted(finished));report()
print(json.dumps(report(),indent=2))

## Consolidação posterior
Os lotes GZIP contêm `records`, com `text`, `sha256`, `proxy_tokens`, `language`, `category`, revisão e origem. Mesmo formato básico do motor de código. Não misturar problemas sintéticos com textos científicos sem manter a classificação. A deduplicação global com v8, Gutenberg e código fica para consolidação; a deduplicação aqui é exata após normalização de espaços e NFC. Não há prova de ausência de contaminação de benchmarks ou correção de soluções. As cotas, filtro temático e licença por registro podem resultar em déficit; o relatório mostra esse déficit sem preenchê-lo com dados não elegíveis. A leitura remota usa timeouts; se houver erro, reiniciar todas as células. A camada de commit protege contra interrupção ordinária, não garante durabilidade física do Drive.